In [1]:
!pip install numerapi --quiet

## Imports

In [3]:
pip install pytorch_tabular

Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys
import numpy as np
import functools

if not hasattr(np, 'rec'):
    from numpy import records
    import types
    sys.modules["numpy.rec"] = records
    np.rec = records
torch.load = functools.partial(torch.load, weights_only=False)
from numerapi import NumerAPI
import pandas as pd
from pytorch_tabular import TabularModel
from pytorch_tabular.models import GANDALFConfig
import numpy
import torch
from torch.utils.data import Dataset, DataLoader
import json
import gc
from rich import print
import json
from rich.table import Table
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)
import typing
import omegaconf
import pyarrow.parquet as pq

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## Exploring the data

In [5]:
train_sample = pd.read_parquet('/kaggle/input/notebooks/svendaj/numerai-data/v5.2/train.parquet').head(1000)
table = Table(title="[bold blue]Numerai Data Audit[/bold blue]")
table.add_column("Metric", style="cyan")
table.add_column("Value", style="magenta")
table.add_row("Total Rows (Sample)", str(len(train_sample)))
table.add_row("Total Features", str(len([c for c in train_sample.columns if "feature" in c])))
table.add_row("Target Name", "target_ender_20")
cat_cols = train_sample.select_dtypes(include=['object']).columns.tolist()
table.add_row("Object-type Columns", str(cat_cols))
print(table)
first_feature = [c for c in train_sample.columns if "feature" in c][0]
print(f"\n[bold green]Stats for {first_feature}:[/bold green]")
print(train_sample[first_feature].describe())

              Numerai Data Audit              
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric              ┃ Value                ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ Total Rows (Sample) │ 1000                 │
│ Total Features      │ 2748                 │
│ Target Name         │ target_ender_20      │
│ Object-type Columns │ ['era', 'data_type'] │
└─────────────────────┴──────────────────────┘

Stats for feature_shaded_hallucinatory_dactylology:

count    1000.000000
mean        1.975000
std         1.422814
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         4.000000
Name: feature_shaded_hallucinatory_dactylology, dtype: float64

In [6]:
print(train_sample.isna().sum().sum())

15

In [7]:
categorical_features = train_sample.select_dtypes(include=['object', 'category'])

if not categorical_features.empty:
    print(f"[bold yellow]Found {len(categorical_features.columns)} Categorical Columns:[/bold yellow]")
    for col in categorical_features.columns:
        unique_count = train_sample[col].nunique()
        print(f" • [cyan]{col}[/cyan]: {unique_count} unique values")
else:
    print("[bold red]No categorical features found. Data is likely pre-normalized (Numerical).[/bold red]")

Found 2 Categorical Columns:

• era: 1 unique values

• data_type: 1 unique values

## PyTorch Models

### GANDALF Model

In [12]:
import typing
import omegaconf
from omegaconf.base import ContainerMetadata, Metadata
from omegaconf.dictconfig import DictConfig
from omegaconf.listconfig import ListConfig
from omegaconf.nodes import AnyNode, EnumNode

torch.serialization.add_safe_globals([
    ContainerMetadata,
    Metadata,
    DictConfig,
    ListConfig,
    AnyNode,
    EnumNode,
    typing.Any
])

In [14]:
# data_config = DataConfig(
#     target=["target_ender_20"],
#     continuous_cols=final_feature_list,
#     categorical_cols=[],
#     num_workers=0,
#     pin_memory=False,
# )

# use_gpu = torch.cuda.is_available()
# print(f"Is GPU available? {use_gpu}")

# trainer_config = TrainerConfig(
#     batch_size=128,
#     max_epochs=150,
#     accelerator="gpu" if use_gpu else "cpu",
#     devices=1,
#     gradient_clip_val=0.1,
#     precision="16-mixed",
#     check_val_every_n_epoch=5,
#     early_stopping="valid_loss",
#     early_stopping_patience=15,
#     early_stopping_mode="min",
#     checkpoints="valid_loss",
#     checkpoints_path="saved_models",
#     checkpoints_mode="min",
#     load_best=True,
#     accumulate_grad_batches=4
# )

# optimizer_config = OptimizerConfig(
#     optimizer="AdamW",
#     optimizer_params={"weight_decay": 1e-5},
#     lr_scheduler="ReduceLROnPlateau",
#     lr_scheduler_params={"factor": 0.5, "patience": 5, "min_lr": 1e-6},
# )

# model_config = GANDALFConfig(
#     task="regression",
#     gflu_stages=6,
#     gflu_feature_init_sparsity=0.3,
#     gflu_dropout=0.15,
#     learning_rate=1e-3,
#     batch_norm_continuous_input=True,
# )

In [15]:
# gandalf_model = TabularModel(
#     data_config = data_config,
#     model_config = model_config,
#     optimizer_config = optimizer_config,
#     trainer_config = trainer_config,
#     verbose = True
# )

In [16]:
# print(validation_dataset['target_ender_20'].isna().sum())

In [17]:
# validation_dataset = validation_dataset.dropna(subset=['target_ender_20'])

In [18]:
# continuous_cols = train_dataset.select_dtypes(include='number').columns.tolist()

# for df in [train_dataset, validation_dataset]:
#     df[continuous_cols] = df[continuous_cols].astype(float)

In [ ]:
import numpy as np
import pandas as pd
import gc
import torch
import json
import pyarrow.parquet as pq
from scipy.linalg import lstsq
from pytorch_tabular import TabularModel
from pytorch_tabular.models import GANDALFConfig
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig

# --- 1. MEMORY-EFFICIENT DATA LOADER ---
def load_parquet_subset(file_name, feature_list, nrows=None):
    path = f'/kaggle/input/notebooks/svendaj/numerai-data/v5.2/{file_name}'
    parquet_file = pq.ParquetFile(path)
    
    schema = parquet_file.schema_arrow
    all_columns = [field.name for field in schema]
    load_cols = feature_list.copy()
    for col in ['target_ender_20', 'era']:
        if col in all_columns:
            load_cols.append(col)
    
    rows_needed = nrows if nrows is not None else float('inf')
    accumulated_rows = 0
    chunks = []
    
    for i in range(parquet_file.num_row_groups):
        chunk = parquet_file.read_row_group(i, columns=load_cols).to_pandas()
        if nrows is not None:
            take = min(len(chunk), rows_needed - accumulated_rows)
            chunk = chunk.iloc[:take]
        
        num_cols = chunk.select_dtypes(include='number').columns
        chunk[num_cols] = chunk[num_cols].astype('float32')
        chunks.append(chunk)
        
        accumulated_rows += len(chunk)
        if nrows is not None and accumulated_rows >= rows_needed:
            break
            
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    return df

# --- 2. VECTORIZED NEUTRALIZATION ---
def neutralize(df, prediction_col, feature_cols, proportion=1.0):
    def _neutralize_era(sub_df):
        y = sub_df[prediction_col].values.reshape(-1, 1)
        X = sub_df[feature_cols].values
        X = np.hstack([X, np.ones((len(X), 1))])
        
        weights, _, _, _ = lstsq(X, y)
        return (y - proportion * (X @ weights)).flatten()

    return df.groupby("era", group_keys=False)[prediction_col].transform(_neutralize_era)

# --- 3. DATA & MODEL INITIALIZATION ---
with open('/kaggle/input/notebooks/svendaj/numerai-data/v5.2/features.json', 'r') as f:
    feature_metadata = json.load(f)

# Using 'medium' features for better memory/performance balance on Kaggle
features_to_use = feature_metadata["feature_sets"]["medium"] 

train_df = load_parquet_subset("train.parquet", features_to_use, 400_000)
val_df = load_parquet_subset("validation.parquet", features_to_use, 200_000)
val_df = val_df.dropna(subset=['target_ender_20'])

data_config = DataConfig(
    target=["target_ender_20"],
    continuous_cols=features_to_use,
    categorical_cols=[],
)

model_config = GANDALFConfig(
    task="regression",
    gflu_stages=2, 
    gflu_dropout=0.2,
    learning_rate=1e-3,
    gflu_feature_init_sparsity=0.5
)

trainer_config = TrainerConfig(
    batch_size=256,
    max_epochs=150,
    accelerator="auto", 
    precision="16-mixed",
    check_val_every_n_epoch=5,
    early_stopping_patience=5,
    accumulate_grad_batches=4
)

optimizer_config = OptimizerConfig(optimizer="AdamW")

model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config
)

# Run Training
history = model.fit(train=train_df, validation=val_df)

2026-03-04 10:01:58,350 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-03-04 10:01:58,413 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-04 10:02:00,328 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-04 10:02:06,890 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: GANDALFModel
2026-03-04 10:02:07,079 - {pytorch_tabular.models.gandalf.gandalf:109} - INFO - Data Aware Initialization of T0
2026-03-04 10:02:07,686 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda),

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ GANDALFBackbone  │  7.3 M │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer │  1.6 K │ train │     0 │
│ 2 │ _head            │ Sequential       │    782 │ train │     0 │
│ 3 │ loss             │ MSELoss          │      0 │ train │     0 │
└───┴──────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 7.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.3 M                                                                                                
Total estimated model params size (MB): 29                                                                         
Modules in train mode: 19                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.

In [ ]:
def neutralize(df, prediction_col, feature_cols, proportion=1.0):
    """
    Corrected neutralization: passes the full DataFrame chunk to access both
    the prediction column and the feature columns.
    """
    def _neutralize_era(sub_df):
        # sub_df is the DataFrame for a single era
        y = sub_df[prediction_col].values.reshape(-1, 1)
        X = sub_df[feature_cols].values
        X = np.hstack([X, np.ones((len(X), 1))])
        
        # Linear least-squares to find the feature exposure
        weights, _, _, _ = lstsq(X, y)
        
        # Subtract the explained portion (proportion * exposure)
        neutral_values = (y - proportion * (X @ weights)).flatten()
        
        # Return as a Series with the original index to maintain alignment
        return pd.Series(neutral_values, index=sub_df.index)

    # We apply the function to the whole GroupBy object (not just one column)
    return df.groupby("era", group_keys=False).apply(_neutralize_era)

In [ ]:
# --- 4. PREDICTION & NEUTRALIZATION ---

# 1. Generate the raw predictions
raw_preds = model.predict(val_df)

# 2. Identify the prediction column name
target_col = "target_ender_20"
pred_col_name = f"{target_col}_prediction"

# 3. FIX: Add the prediction column to your ORIGINAL val_df
val_df[pred_col_name] = raw_preds[pred_col_name].values

print(f"Neutralizing {pred_col_name} against {len(features_to_use)} features...")

# 4. Apply Neutralization

val_df["neutral_scores"] = neutralize(
    val_df, 
    prediction_col=pred_col_name, 
    feature_cols=features_to_use, 
    proportion=0.5
)

# 5. Final Ranking (scale to 0-1)
val_df["prediction"] = val_df["neutral_scores"].rank(pct=True)

print("Neutralization successful. Features found and correlations removed.")

In [ ]:
import numpy as np
import pandas as pd

def calculate_numerai_metrics(df, pred_col, target_col):
    """
    Calculates the primary metrics used to judge a Numerai submission.
    """
    # 1. Per-Era Correlation (Spearman)
    # This measures the rank-order similarity between your prediction and the target
    def spearman_corr(sub_df):
        return sub_df[pred_col].corr(sub_df[target_col], method='spearman')

    era_corrs = df.groupby("era").apply(spearman_corr)

    # 2. Key Metrics Calculations
    mean_corr = era_corrs.mean()
    std_corr = era_corrs.std()
    # Sharpe Ratio: How consistent is your return relative to the risk (volatility)?
    sharpe = mean_corr / std_corr if std_corr != 0 else 0
    
    # 3. Max Drawdown
    # Measures the biggest peak-to-trough drop in cumulative correlation
    cumulative_corr = era_corrs.cumsum()
    rolling_max = cumulative_corr.expanding().max()
    drawdown = cumulative_corr - rolling_max
    max_drawdown = drawdown.min()

    return {
        "Mean Corr": mean_corr,
        "Sharpe": sharpe,
        "Max Drawdown": max_drawdown,
        "Era Corrs": era_corrs
    }

# --- USAGE ---
# Calculate metrics for your neutralized predictions
metrics = calculate_numerai_metrics(val_df, "prediction", "target_ender_20")

print(f"--- VALIDATION RESULTS ---")
print(f"Mean Correlation: {metrics['Mean Corr']:.5f}")
print(f"Sharpe Ratio:     {metrics['Sharpe']:.5f}")
print(f"Max Drawdown:     {metrics['Max Drawdown']:.5f}")

# Plotting the performance over time
metrics['Era Corrs'].cumsum().plot(title="Cumulative Correlation (Validation Set)", figsize=(10, 5))

In [ ]:
# history = gandalf_model.fit(train=train_dataset, validation=validation_dataset)

In [ ]:
log_dir = gandalf_model.trainer.logger.log_dir
log_dir

In [ ]:
!pip install tbparse -q

from tbparse import SummaryReader

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
reader = SummaryReader(log_dir)
df = reader.scalars
train_loss = df[df['tag'] == 'train_loss']
val_loss = df[df['tag'] == 'valid_loss']

plt.figure(figsize=(10, 5))
plt.plot(train_loss['step'], train_loss['value'], label='Train Loss')
plt.plot(val_loss['step'], val_loss['value'], label='Val Loss')
plt.title('GANDALF - Historia Trenowania')
plt.legend()
plt.show()

In [ ]:
def evaluate_model(df, prediction_col, target_col="target_ender_20"):
    era_correlations = df.groupby("era").apply(
        lambda x: np.corrcoef(x[target_col], x[prediction_col])[0, 1]
    )
    
    mean_corr = era_correlations.mean()
    std_corr = era_correlations.std()
    sharpe = mean_corr / std_corr
    
    cumulative_corr = era_correlations.cumsum()
    rolling_max = cumulative_corr.expanding().max()
    drawdown = cumulative_corr - rolling_max
    max_drawdown = drawdown.min()
    
    stats_table = Table(title="[bold green]Model Performance Metrics[/bold green]")
    stats_table.add_column("Metric", style="cyan")
    stats_table.add_column("Value", style="magenta")
    
    stats_table.add_row("Mean Era Correlation", f"{mean_corr:.5f}")
    stats_table.add_row("Sharpe Ratio", f"{sharpe:.4f}")
    stats_table.add_row("Standard Deviation", f"{std_corr:.5f}")
    stats_table.add_row("Max Drawdown", f"{max_drawdown:.5f}")
    
    print(stats_table)
    
    return era_correlations

In [ ]:
predictions = model.predict(val_df)

val_df["prediction"] = prediction_results["target_ender_20_prediction"]

era_scores = evaluate_model(val_df, "prediction")

plt.figure(figsize=(12, 5))
era_scores.plot(kind='bar', color='skyblue', title='Correlation by Era')
plt.axhline(y=era_scores.mean(), color='r', linestyle='--', label=f'Mean: {era_scores.mean():.4f}')
plt.legend()
plt.xticks([])
plt.show()

In [ ]:
import lightgbm as lgb

X_train = train_dataset[small_features]
y_train = train_dataset['target_ender_20']
X_val = validation_dataset[small_features]
y_val = validation_dataset['target_ender_20']

lgb_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=31,
    colsample_bytree=0.1, 
    random_state=42,
    verbosity=-1,
    device="gpu" if torch.cuda.is_available() else "cpu"
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='l2',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100)
    ]
)

In [ ]:
# GANDALF predictions
gandalf_pred_df = model.predict(validation_dataset)

# Find the prediction column – typically it's 'target_ender_20_prediction'
possible_pred_cols = ['target_ender_20_prediction', 'prediction', gandalf_pred_df.columns[0]]
pred_col = next((col for col in possible_pred_cols if col in gandalf_pred_df.columns), gandalf_pred_df.columns[0])
print(f"Using prediction column: {pred_col}")

val_preds_gandalf = gandalf_pred_df[pred_col].values

# Add GANDALF predictions to the validation DataFrame
validation_dataset['gandalf_pred'] = val_preds_gandalf

# LightGBM predictions (assuming X_val is aligned with validation_dataset.df)
val_preds_lgb = lgb_model.predict(X_val)
validation_dataset['lgb_pred'] = val_preds_lgb

# Compute percent ranks
gandalf_rank = validation_dataset['gandalf_pred'].rank(pct=True)
lgb_rank = validation_dataset['lgb_pred'].rank(pct=True)
validation_dataset['ensemble_rank'] = (gandalf_rank + lgb_rank) / 2

# Evaluate each model using your custom function
print("\n[cyan]Results of GANDALF:[/cyan]")
evaluate_model(validation_dataset, "gandalf_pred")

print("\n[cyan]Results of LightGBM:[/cyan]")
evaluate_model(validation_dataset, "lgb_pred")

print("\n[bold green]Results of the ensemble:[/bold green]")
evaluate_model(validation_dataset, "ensemble_rank")

In [ ]:
model.save_model("gandalf_model")
gandalf_model = TabularModel.load_model("gandalf_model")

In [ ]:
submission_df = predictions[['target_ender_20_prediction']].copy()
submission_df.index = live_dataset.index
submission_df.columns = ['target_ender_20_prediction']

In [ ]:
all_features = [c for c in live_dataset.columns if c.startswith("feature")]

def neutralize(df, prediction_col, feature_cols, proportion=1.0):
    def neutralize_era(sub_df):
        y = sub_df[prediction_col].values.reshape(-1, 1)
        X = sub_df[feature_cols].values
        
        X = np.hstack([X, np.ones((len(X), 1))])
        
        projection = X @ np.linalg.lstsq(X, y, rcond=None)[0]
        
        neutralized_y = y - (proportion * projection)
        return neutralized_y.flatten()

    return df.groupby("era", group_keys=False).apply(
        lambda x: pd.Series(neutralize_era(x), index=x.index)
    )

submission_df["target_ender_20_prediction"] = neutralize(
    submission_df, 
    "target_ender_20_prediction", 
    all_features, 
    proportion=0.5
)

submission_df["target_ender_20_prediction"] = submission_df["target_ender_20_prediction"].rank(pct=True)

In [ ]:
submission_df['target_ender_20_prediction'].hist(bins=50)
plt.title("Rozkład prognoz")
plt.show()

In [ ]:
import numpy as np

exposures = []
live_df = live_dataset

for col in all_features:
    cor = np.corrcoef(
        submission_df['target_ender_20_prediction'].values, 
        live_df[col].values
    )[0, 1]
    exposures.append(cor)

max_exposure = np.max(np.abs(exposures))
print(f"Maksymalna ekspozycja na pojedynczą cechę: {max_exposure:.4f}")

In [ ]:
final_submission = submission_df.rename(columns={'target_ender_20_prediction': 'prediction'})
final_submission.to_csv("submission.csv", index=True)